# 06_pretokenize

『밑바닥부터 시작하는 딥러닝 ❻』 실습 코드 — 원본: `ch01/06_pretokenize.py`

셀을 위에서부터 차례대로 실행하세요.

In [ ]:
from collections import defaultdict
import regex as re
from tqdm import tqdm

In [ ]:
def pretokenize(text):
    # GPT-2에서 사용하는 정규 표현식 패턴
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)

In [ ]:
def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

In [ ]:
def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids

In [ ]:
def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 1단계: 특수 토큰을 기준으로 분할
    texts = input_text.split(end_token)

    # 2단계: 각 텍스트 조각을 사전 토큰화
    ids_list = []
    for text in texts:
        for pretoken in pretokenize(text):  # 사전 토큰화
            ids_list.append(list(pretoken.encode("utf-8")))  # ID열로 변환

    # ==== 나머지는 원래 코드와 동일(tqdm만 추가) ====
    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):  # tqdm으로 진행률 표시
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        if not counts:
            break

        best_pair = max(counts, key=counts.get)
        # best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules

In [ ]:
class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text, show_progress=False):  # 진행률 표시 인수 추가
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # # show_progress가 True면 tqdm으로 진행률 표시
        texts = tqdm(texts, desc="Encoding") if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 각 사전 토큰을 BPE로 인코딩
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text

In [ ]:
# 사전 토큰화를 적용한 BPE 학습
sample_text = "Say hello! Why hello? Just hello.<|endoftext|>Good morning!"

In [ ]:
merge_rules = train_bpe(sample_text, vocab_size=270)
tokenizer = BPETokenizer(merge_rules)

In [ ]:
# 인코딩과 디코딩
text = "Say hello!"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

In [ ]:
print(ids)
print(decoded)

In [ ]:
# 각 토큰 ID를 개별적으로 디코딩하여 확인
for token_id in ids:
    print(f"{token_id} -> '{tokenizer.decode([token_id])}'")